# Haiku MIL Classification (patient-disjoint)

**Project Name:** Haiku

## Purpose
- Publication-ready notebook for reproducible evaluation.
- 5-fold **patient-wise** `StratifiedGroupKFold` CV (no patient leakage).
  patient_id is derived from `SAMPLE_LABEL` by stripping a trailing
  `_<digit>` replicate suffix (e.g. `CX-18A_1` -> `CX-18A`).
- Hyperparameters retuned per task with the patient-wise sweep
  (`mil_sweep.py --tasks cls1 cls2 --max_rounds 3`).

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
SAMPLES_JSON   = HAIKU_ROOT / 'overlap_samples_final.json'
OUTPUT_DIR     = HAIKU_ROOT / 'downstream' / 'figs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from haiku import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from transformers import BertTokenizer
import pandas as pd
import numpy as np
import torch.nn.functional as F
import json
import sys


codex_embedding = torch.load(EMBEDDINGS_DIR / 'new_codex_embedding.pt')

virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'new_virtual_codex_embedding.pt')

region_label = torch.load(EMBEDDINGS_DIR / 'new_region_label.pt')

print(codex_embedding.shape)

sample_ids = list(json.load(open(SAMPLES_JSON)).keys())

ref_ids = sorted(sample_ids)

# Build bags of embeddings for each patient
# The mapping from region_label[i] -> ref_ids[region_label[i]] will give you a region/acquisition ID
patient_he_bags = dict()        # patient_id: [he_embedding_tensors]
patient_codex_bags = dict()  # patient_id: [codex_embedding_tensors]
patient_virt_bags = dict()   # patient_id: [virtual_codex_embedding_tensors]
patient_musk_bags = dict()   # patient_id: [musk_embedding_tensors]
patient_labels = dict()
patient_concat_bags = dict()    # patient_id: label (list or scalar per patient, depending)

for idx, region_l in enumerate(region_label):
    # Lookup the patient_id for this region/acquisition

    region_id = ref_ids[region_l]

    # Group embeddings by patient
    #patient_he_bags.setdefault(region_id, []).append(he_embedding[idx])
    patient_codex_bags.setdefault(region_id, []).append(codex_embedding[idx])
    patient_virt_bags.setdefault(region_id, []).append(virtual_codex_embedding[idx])
    #patient_musk_bags.setdefault(region_id, []).append(musk_embedding[idx])
    #patient_concat_bags.setdefault(region_id, []).append(torch.cat([he_embedding[idx], codex_embedding[idx]], dim=0))

    # Optionally: also maintain region-labels or per-patient labels.
    # Here, defaulting to using the region's integer label.
    patient_labels.setdefault(region_id, []).append(region_label[idx])

# Optionally, if you want a "single label per patient" for classification (e.g. majority, first, or a function):
# Example (majority label per patient):
from collections import Counter
patient_majority_labels = {
    pid: Counter(lbls).most_common(1)[0][0]
    for pid, lbls in patient_labels.items()
}

# patient_bags, patient_codex_bags, patient_virt_bags, patient_musk_bags now each map a patient_id -> list of embeddings as bags.
# patient_labels: patient_id -> list of integer region label per patch, or use patient_majority_labels for (patient_id -> majority label).


In [ ]:
import os
import pandas as pd

csv_list = [
    HAIKU_ROOT / 'downstream' / 'Lymphoma_response-to-RCHOP.csv',
    HAIKU_ROOT / 'downstream' / 'Melanoma_response-to-immunotherapy.csv',
    HAIKU_ROOT / 'downstream' / 'CRC_terminal_survival.csv',
]


need_new_acq_ids = []

df_list = []

for i in csv_list:
    df = pd.read_csv(i)
    new_acq_ids = df['ACQUISITION_ID'].tolist()
    df_list.append(df)


In [ ]:
df_list[1].columns

In [ ]:
df_list[2]['description'].unique()

In [ ]:
df_list[2]['description'].shape, df_list[1]['description'].shape


In [ ]:
df_list[1]['description'].unique()

In [ ]:
survial_length_dict = {}
survial_status_dict = { }
response_dict = {}
treatment_dict = {}

In [ ]:
response_dict['0'] = 'Response-binary'
treatment_dict['0'] = 'treatment'

In [ ]:
survial_length_dict['1'] = 'survival'
survial_status_dict['1'] = 'survival_status'
response_dict['1'] = 'Response-binary'
treatment_dict['1'] = 'treatment'

In [ ]:
survial_length_dict['2'] = 'FOLLOW UP (months)'
survial_status_dict['2'] = 'survival_status'
response_dict['2'] = 'Outcome-binary'
treatment_dict['2'] = 'treatment'

In [ ]:
# End-to-end MIL Classification (Binary): 5-fold *patient-wise* CV
# Expected objects already in scope (built by upstream cells):
#   - patient_virt_bags    : {acquisition_id: [emb, ...]}  # Baseline (VirTues)
#   - patient_codex_bags   : {acquisition_id: [emb, ...]}  # Ours
#   - df_list, sample_ids, response_dict, treatment_dict
#
# Mirrors examples/mil_cls.py (post-leakage-fix):
#   * StratifiedGroupKFold on patient_id (groups), label-stratified.
#   * Patient-aware inner train/val split (no patient leak into val/test).
#   * Cosine LR + patience early-stop + best-state restore.
#   * Per-task BEST_HP from the patient-wise sweep.

import os, json, random, copy, re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Dict, List
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    accuracy_score, f1_score
)

# ----------------------------
# 0) Reproducibility + Device
# ----------------------------
SEED = 42
N_FOLDS = 5

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Keep text editable in SVG/PDF exports
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

# ----------------------------
# Best HPs from PATIENT-WISE sweep (see sweep_results/cls_task_*_sweep_patientwise.json)
# Picked by mean (AUROC + AUPRC) / 2 of "ours" across 5 patient-wise folds.
# ----------------------------
BEST_HP_CLS = {
    1: {"embed_dim": 32, "pool": "attn", "lr": 2e-3, "dropout": 0.1,
        "l2_norm": False, "batch_size": 32, "wd": 1e-4,
        "max_epochs": 50, "patience": 10},   # Melanoma response (IO)
    2: {"embed_dim": 64, "pool": "attn", "lr": 5e-4, "dropout": 0.1,
        "l2_norm": False, "batch_size": 8,  "wd": 1e-4,
        "max_epochs": 50, "patience": 10},   # CRC outcome
}

# ---------------------------------------
# 1) Dataset and collate for padded bags
# ---------------------------------------
def l2_normalize(x, axis=1, eps=1e-12):
    norm = np.linalg.norm(x, ord=2, axis=axis, keepdims=True)
    return x / np.clip(norm, a_min=eps, a_max=None)


class MILDataset(Dataset):
    def __init__(self, emb_dict: Dict[str, List[np.ndarray]],
                 labels: Dict[str, int],
                 bag_ids: List[str],
                 do_l2_norm: bool = True):
        self.ids, self.bags, self.y = [], [], []
        for bid in bag_ids:
            if (bid not in emb_dict) or (bid not in labels):
                continue
            raw = emb_dict[bid]
            if isinstance(raw, np.ndarray) and raw.ndim == 2:
                X = raw.astype(np.float32)
            elif isinstance(raw, torch.Tensor) and raw.ndim == 2:
                X = raw.numpy().astype(np.float32)
            else:
                if len(raw) == 0:
                    continue
                X = np.stack([r.numpy() if torch.is_tensor(r) else np.asarray(r)
                              for r in raw], axis=0).astype(np.float32)
            if do_l2_norm:
                X = l2_normalize(X, axis=1)
            self.ids.append(bid)
            self.bags.append(torch.from_numpy(X))
            self.y.append(int(labels[bid]))
        if len(self.bags) == 0:
            raise ValueError("Empty MILDataset — check dictionaries/split.")
        self.Din = self.bags[0].shape[1]

    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return self.bags[i], torch.tensor(self.y[i], dtype=torch.long), self.ids[i]


def pad_collate(batch):
    bags, ys, ids = zip(*batch)
    B = len(bags)
    Nmax = max(b.shape[0] for b in bags)
    D = bags[0].shape[1]
    X = torch.zeros(B, Nmax, D, dtype=torch.float32)
    M = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        n = b.shape[0]
        X[i, :n] = b
        M[i, :n] = True
    y = torch.stack(ys)
    return X, M, y, list(ids)

# --------------------------------
# 2) MIL Model (Encoder + Pool + Head)
# --------------------------------
class GatedAttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.U = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H)) * torch.sigmoid(self.U(H))).squeeze(-1)
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        return torch.einsum("bn,bnd->bd", A, H), A


class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        return torch.einsum("bn,bnd->bd", A, H), A


class MILClassifier(nn.Module):
    def __init__(self, in_dim, embed_dim=128, pool="attn", dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, embed_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.LayerNorm(embed_dim),
        )
        self.pool_type = pool
        if pool == "gated_attn":
            self.pool = GatedAttnPool(embed_dim)
        elif pool == "attn":
            self.pool = AttnPool(embed_dim)
        else:
            self.pool = None
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, X, mask):
        H = self.encoder(X)
        if self.pool_type in ("attn", "gated_attn"):
            Z, A = self.pool(H, mask)
        elif self.pool_type == "mean":
            Z = (H * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1).to(H.dtype)
            A = None
        else:
            raise ValueError(f"Unknown pool: {self.pool_type}")
        return self.head(Z).squeeze(-1), A

# --------------------------------
# 3) Patient-aware split helpers
# --------------------------------
def build_aid2pid(df):
    """ACQUISITION_ID -> patient_id from SAMPLE_LABEL, stripping trailing
    `_<digit>` replicate suffix (e.g. CX-18A_1 -> CX-18A)."""
    pid = df["SAMPLE_LABEL"].astype(str).map(lambda s: re.sub(r"_\d+$", "", s))
    return dict(zip(df["ACQUISITION_ID"], pid))


def grouped_train_val_split(ids, groups, stratify=None, test_size=0.1, seed=SEED):
    """Patient-aware inner train/val split. Prefers StratifiedGroupKFold when
    feasible, else GroupShuffleSplit. Guarantees disjoint patient groups."""
    ids_arr = np.asarray(ids); groups = np.asarray(groups)
    if len(set(groups)) < 2:
        return ids, ids
    if stratify is not None and len(np.unique(stratify)) >= 2:
        for nsp in (max(2, int(round(1.0 / test_size))), 5, 3, 2):
            try:
                sgkf = StratifiedGroupKFold(n_splits=nsp, shuffle=True, random_state=seed)
                tr_idx, va_idx = next(sgkf.split(ids_arr, stratify, groups=groups))
                return ids_arr[tr_idx].tolist(), ids_arr[va_idx].tolist()
            except ValueError:
                continue
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(gss.split(ids_arr, groups=groups))
    return ids_arr[tr_idx].tolist(), ids_arr[va_idx].tolist()

# --------------------------------
# 4) Train / evaluate routines
# --------------------------------
@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    all_ids, all_y, all_logits = [], [], []
    for X, M, y, ids in loader:
        X, M = X.to(device), M.to(device)
        logits, _ = model(X, M)
        all_ids += ids
        all_y += y.numpy().tolist()
        all_logits += logits.cpu().numpy().tolist()
    y_true = np.array(all_y)
    logits = np.array(all_logits)
    probs = 1.0 / (1.0 + np.exp(-logits))
    return all_ids, y_true, probs, logits


def evaluate_binary(y_true, prob, threshold=0.5):
    if not isinstance(y_true, np.ndarray): y_true = np.array(y_true)
    if not isinstance(prob, np.ndarray): prob = np.array(prob)
    auroc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan
    auprc = average_precision_score(y_true, prob)
    y_pred = (prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    fpr, tpr, _ = roc_curve(y_true, prob) if len(np.unique(y_true)) > 1 else (np.array([0, 1]), np.array([0, 1]), None)
    prec, rec, _ = precision_recall_curve(y_true, prob)
    return {
        "auroc": float(auroc), "auprc": float(auprc), "acc": float(acc), "f1_macro": float(f1m),
        "fpr": fpr.tolist(), "tpr": tpr.tolist(), "prec": prec.tolist(), "rec": rec.tolist()
    }


def train_mil_classifier(emb_dict, labels, train_ids, val_ids, hp, device=device):
    """Cosine LR, patience early-stop, best-state restore. Mirrors
    examples/mil_cls.py:train_model."""
    set_seed(SEED)
    train_ds = MILDataset(emb_dict, labels, train_ids, do_l2_norm=hp["l2_norm"])
    val_ds   = MILDataset(emb_dict, labels, val_ids,   do_l2_norm=hp["l2_norm"])

    train_loader = DataLoader(train_ds, batch_size=hp["batch_size"], shuffle=True,
                              collate_fn=pad_collate, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=hp["batch_size"], shuffle=False,
                              collate_fn=pad_collate, num_workers=0)

    model = MILClassifier(train_ds.Din, hp["embed_dim"], hp["pool"], hp["dropout"]).to(device)

    y_train = np.array([labels[_id] for _id in train_ds.ids])
    pos = (y_train == 1).sum(); neg = (y_train == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    scheduler = CosineAnnealingLR(opt, T_max=hp["max_epochs"], eta_min=hp["lr"] * 0.01)

    best_val_score = -1.0; best_state = None; patience_counter = 0
    for epoch in range(1, hp["max_epochs"] + 1):
        model.train()
        for X, M, y, _ in train_loader:
            X, M, y = X.to(device), M.to(device), y.to(device).float()
            logits, _ = model(X, M)
            loss = criterion(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()

        model.eval()
        all_y, all_p = [], []
        with torch.no_grad():
            for X, M, y, _ in val_loader:
                logits, _ = model(X.to(device), M.to(device))
                all_y.append(y.numpy())
                all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_y = np.concatenate(all_y); all_p = np.concatenate(all_p)
        score = ((roc_auc_score(all_y, all_p) + average_precision_score(all_y, all_p)) / 2
                 if len(np.unique(all_y)) > 1 else 0.0)

        if score > best_val_score:
            best_val_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= hp["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# ---------------------------------------
# 5) Utilities
# ---------------------------------------
def _to_scalar_label(v):
    arr = np.asarray(v)
    if arr.shape == ():
        return int(arr)
    return int(arr.reshape(-1)[0])


def make_loader(emb_dict, labels, ids, hp, batch=256):
    ds = MILDataset(emb_dict, labels, ids, do_l2_norm=hp["l2_norm"])
    return DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=pad_collate)


def interpolate_mean_curve(xs, ys_list, grid):
    mats = []
    for x, y in zip(xs, ys_list):
        x = np.asarray(x); y = np.asarray(y)
        order = np.argsort(x)
        mats.append(np.interp(grid, x[order], y[order]))
    mat = np.vstack(mats)
    return mat.mean(0), mat.std(0)

# ---------------------------------------
# 6) Five-fold patient-wise CV (Baseline vs Ours)
# ---------------------------------------
def run_mil_cls_cv5(baseline_embeddings, our_embeddings, labels_dict, aid2pid,
                    hp, device=device, seed=SEED):
    """Patient-wise 5-fold CV. Mirrors examples/mil_cls.py:run_cv5.

    `aid2pid` maps acquisition_id -> patient_id. Folds are produced by
    StratifiedGroupKFold on `groups=patient_id`, label-stratified."""
    set_seed(seed)

    all_ids = sorted(
        set(labels_dict) & set(baseline_embeddings) & set(our_embeddings) & set(aid2pid)
    )
    y_all   = np.array([_to_scalar_label(labels_dict[i]) for i in all_ids], dtype=int)
    groups  = np.array([aid2pid[i] for i in all_ids])
    print(f"  Patient-wise CV: {len(all_ids)} acquisitions, "
          f"{len(set(groups))} patients.")

    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    per_fold_metrics = {"baseline": [], "ours": []}
    base_probs_all, base_true_all = [], []
    ours_probs_all, ours_true_all = [], []
    roc_fprs_b, roc_tprs_b = [], []
    roc_fprs_o, roc_tprs_o = [], []
    pr_recs_b, pr_precs_b  = [], []
    pr_recs_o, pr_precs_o  = [], []

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(all_ids, y_all, groups=groups), start=1):
        train_ids = [all_ids[i] for i in tr_idx]
        test_ids  = [all_ids[i] for i in te_idx]
        train_groups = [groups[i] for i in tr_idx]

        y_tr = np.array([labels_dict[i] for i in train_ids], dtype=int)
        if len(train_ids) < 10 or len(np.unique(y_tr)) < 2:
            continue

        # Patient-aware inner split: no patient leak across train/val/test.
        tr_ids, va_ids = grouped_train_val_split(
            train_ids, train_groups, stratify=y_tr, test_size=0.1, seed=seed,
        )
        tr_pids = {aid2pid[i] for i in tr_ids}
        va_pids = {aid2pid[i] for i in va_ids}
        te_pids = {aid2pid[i] for i in test_ids}
        assert not ((tr_pids | va_pids) & te_pids), f"patient leakage in fold {fold_idx}"
        print(f"  Fold {fold_idx}/{N_FOLDS}: train={len(tr_ids)} (pat={len(tr_pids)}), "
              f"val={len(va_ids)} (pat={len(va_pids)}), "
              f"test={len(test_ids)} (pat={len(te_pids)})")

        base_model = train_mil_classifier(
            baseline_embeddings, labels_dict, tr_ids, va_ids, hp, device=device)
        ours_model = train_mil_classifier(
            our_embeddings, labels_dict, tr_ids, va_ids, hp, device=device)

        base_loader = make_loader(baseline_embeddings, labels_dict, test_ids, hp)
        ours_loader = make_loader(our_embeddings,    labels_dict, test_ids, hp)

        _, yt_b, pt_b, _ = predict_model(base_model, base_loader, device)
        _, yt_o, pt_o, _ = predict_model(ours_model, ours_loader, device)

        m_b = evaluate_binary(yt_b, pt_b); m_o = evaluate_binary(yt_o, pt_o)
        per_fold_metrics["baseline"].append({"fold": fold_idx, **m_b})
        per_fold_metrics["ours"].append({"fold": fold_idx, **m_o})

        base_probs_all.append(pt_b); base_true_all.append(yt_b)
        ours_probs_all.append(pt_o); ours_true_all.append(yt_o)

        roc_fprs_b.append(np.array(m_b["fpr"])); roc_tprs_b.append(np.array(m_b["tpr"]))
        roc_fprs_o.append(np.array(m_o["fpr"])); roc_tprs_o.append(np.array(m_o["tpr"]))
        pr_recs_b.append(np.array(m_b["rec"])); pr_precs_b.append(np.array(m_b["prec"]))
        pr_recs_o.append(np.array(m_o["rec"])); pr_precs_o.append(np.array(m_o["prec"]))

    base_probs_all = np.concatenate(base_probs_all); base_true_all = np.concatenate(base_true_all)
    ours_probs_all = np.concatenate(ours_probs_all); ours_true_all = np.concatenate(ours_true_all)
    pooled_base = evaluate_binary(base_true_all, base_probs_all)
    pooled_ours = evaluate_binary(ours_true_all, ours_probs_all)

    results = {
        "fold_metrics": per_fold_metrics,
        "pooled": {"baseline": pooled_base, "ours": pooled_ours},
        "hyperparameters": hp,
    }

    # ----------------- Plots -----------------
    mean_fpr = np.linspace(0, 1, 400)
    mean_tpr_b, std_tpr_b = interpolate_mean_curve(roc_fprs_b, roc_tprs_b, mean_fpr)
    mean_tpr_o, std_tpr_o = interpolate_mean_curve(roc_fprs_o, roc_tprs_o, mean_fpr)
    plt.figure(figsize=(6.2, 5.2))
    for fpr, tpr in zip(roc_fprs_b, roc_tprs_b): plt.plot(fpr, tpr, alpha=0.25, lw=1)
    for fpr, tpr in zip(roc_fprs_o, roc_tprs_o): plt.plot(fpr, tpr, alpha=0.25, lw=1)
    plt.fill_between(mean_fpr, np.clip(mean_tpr_b - std_tpr_b, 0, 1), np.clip(mean_tpr_b + std_tpr_b, 0, 1), alpha=0.15, label="Baseline (±1σ)")
    plt.fill_between(mean_fpr, np.clip(mean_tpr_o - std_tpr_o, 0, 1), np.clip(mean_tpr_o + std_tpr_o, 0, 1), alpha=0.15, label="Ours (±1σ)")
    fpr_b, tpr_b = np.array(pooled_base["fpr"]), np.array(pooled_base["tpr"])
    fpr_o, tpr_o = np.array(pooled_ours["fpr"]), np.array(pooled_ours["tpr"])
    plt.plot(fpr_b, tpr_b, lw=2.5, label=f"Baseline pooled (AUROC={pooled_base['auroc']:.3f})")
    plt.plot(fpr_o, tpr_o, lw=2.5, label=f"Ours pooled (AUROC={pooled_ours['auroc']:.3f})")
    plt.plot([0, 1], [0, 1], '--', alpha=0.4)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC — 5-fold patient-wise CV")
    plt.legend(framealpha=0.9); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

    mean_rec = np.linspace(0, 1, 400)
    mean_prec_b, std_prec_b = interpolate_mean_curve(pr_recs_b, pr_precs_b, mean_rec)
    mean_prec_o, std_prec_o = interpolate_mean_curve(pr_recs_o, pr_precs_o, mean_rec)
    prev = base_true_all.mean() if base_true_all.size > 0 else 0.5
    plt.figure(figsize=(6.2, 5.2))
    for rc, pr in zip(pr_recs_b, pr_precs_b): plt.plot(rc, pr, alpha=0.25, lw=1)
    for rc, pr in zip(pr_recs_o, pr_precs_o): plt.plot(rc, pr, alpha=0.25, lw=1)
    plt.fill_between(mean_rec, np.clip(mean_prec_b - std_prec_b, 0, 1), np.clip(mean_prec_b + std_prec_b, 0, 1), alpha=0.15, label="Baseline (±1σ)")
    plt.fill_between(mean_rec, np.clip(mean_prec_o - std_prec_o, 0, 1), np.clip(mean_prec_o + std_prec_o, 0, 1), alpha=0.15, label="Ours (±1σ)")
    plt.plot(np.array(pooled_base["rec"]), np.array(pooled_base["prec"]),
             lw=2.5, label=f"Baseline pooled (AUPRC={pooled_base['auprc']:.3f})")
    plt.plot(np.array(pooled_ours["rec"]), np.array(pooled_ours["prec"]),
             lw=2.5, label=f"Ours pooled (AUPRC={pooled_ours['auprc']:.3f})")
    plt.hlines(prev, 0, 1, linestyles="--", alpha=0.4, label=f"No-skill (p={prev:.2f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision–Recall — 5-fold patient-wise CV")
    plt.legend(framealpha=0.9); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

    aurocs_b = [m["auroc"] for m in per_fold_metrics["baseline"]]
    aurocs_o = [m["auroc"] for m in per_fold_metrics["ours"]]
    auprcs_b = [m["auprc"] for m in per_fold_metrics["baseline"]]
    auprcs_o = [m["auprc"] for m in per_fold_metrics["ours"]]
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=False)
    axes[0].boxplot([aurocs_b, aurocs_o], labels=["Baseline", "Ours"], showmeans=True)
    axes[0].set_title("AUROC (5-fold patient-wise)"); axes[0].set_ylim(0, 1)
    axes[1].boxplot([auprcs_b, auprcs_o], labels=["Baseline", "Ours"], showmeans=True)
    axes[1].set_title("AUPRC (5-fold patient-wise)"); axes[1].set_ylim(0, 1)
    plt.suptitle("MIL Classification — 5-fold patient-wise CV boxplots")
    plt.tight_layout(); plt.show()

    return results, base_probs_all, ours_probs_all

# ---------------------------------------
# 7) Building labels from df_list and running CV
# ---------------------------------------
def build_labels_from_df(df, sample_ids, response_key, treatment_key):
    region_labels_response = {}
    region_labels_diagnosis = {}
    for region_id in sample_ids:
        if region_id in df['ACQUISITION_ID'].values:
            response_val = df.loc[df['ACQUISITION_ID'] == region_id, response_key].values
            treat_val    = df.loc[df['ACQUISITION_ID'] == region_id, treatment_key].values
            region_labels_response[region_id]  = _to_scalar_label(response_val)
            region_labels_diagnosis[region_id] = treat_val
    return region_labels_response, region_labels_diagnosis

# -------- Main loop over df_list (per-task BEST_HP from patient-wise sweep) --------
probs_all_res = []
ours_probs_all_res = []
results_per_task = {}

for i, df in enumerate(df_list[1:]):  # df_list[1] -> task 1 (Melanoma), df_list[2] -> task 2 (CRC)
    task_id = i + 1
    response_key  = response_dict[str(task_id)]
    treatment_key = treatment_dict[str(task_id)]

    labels_response, _ = build_labels_from_df(df, sample_ids, response_key, treatment_key)
    aid2pid = build_aid2pid(df)
    hp = BEST_HP_CLS[task_id]

    print(f"\n{'='*60}")
    print(f"Task {task_id}: {response_key} ({len(labels_response)} labeled bags)")
    print(f"HP: embed_dim={hp['embed_dim']}, pool={hp['pool']}, lr={hp['lr']}, "
          f"dropout={hp['dropout']}, l2_norm={hp['l2_norm']}, bs={hp['batch_size']}, wd={hp['wd']}")
    print('=' * 60)

    results, base_probs_all, ours_probs_all = run_mil_cls_cv5(
        baseline_embeddings=patient_virt_bags,
        our_embeddings=patient_codex_bags,
        labels_dict=labels_response,
        aid2pid=aid2pid,
        hp=hp,
        device=device,
        seed=SEED,
    )

    results_per_task[f"task_{task_id}"] = results
    probs_all_res.append(base_probs_all)
    ours_probs_all_res.append(ours_probs_all)
